El objetivo aquí es verificar que los datos se han cargado correctamente y tener una primera idea de su estructura.

## 1. Carga y combinación de datos de viajes

In [1]:
import pandas as pd
import glob
import numpy as np

# Se crea una variable de texto que guarda la ruta a la carpeta de datos
# Se usa glob para buscar dentro de la carpeta ../data/. Busca cualquier archivo (*) que contenga la palabra tripdata y que termine en .csv
path_viajes = '../data/'
archivos_csv_viajes = glob.glob(path_viajes + "*tripdata*.csv")

# Diccionario que define explícitamente los tipos para columnas problemáticas - ayuda a evitar el mensaje DtypeWarning
tipos_de_datos = {'ride_id': str, 'start_station_id': str, 'end_station_id': str}

# Carga los archivos usando la opción dtype
lista_de_dfs = []
for archivo in archivos_csv_viajes:
    # Añadimos el parámetro 'dtype'
    df_temporal = pd.read_csv(archivo, dtype=tipos_de_datos)
    lista_de_dfs.append(df_temporal)

# Concatena todos los DataFrames
df_trips = pd.concat(lista_de_dfs, ignore_index=True)

df_trips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3168271 entries, 0 to 3168270
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             object 
 1   rideable_type       object 
 2   started_at          object 
 3   ended_at            object 
 4   start_station_name  object 
 5   start_station_id    object 
 6   end_station_name    object 
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       object 
dtypes: float64(4), object(9)
memory usage: 314.2+ MB


In [2]:
# Comprobamos las primeras 5 filas del historial de viajes
df_trips.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,0AE5FB7DFCB8E5BE,classic_bike,2025-03-11 17:33:37.322,2025-03-11 17:36:58.910,Chauncey St & Stuyvesant Ave,4196.05,Lewis Ave & Decatur St,4237.01,40.680120,-73.931680,40.681460,-73.934903,member
1,0F8A4821A038CDBE,electric_bike,2025-03-04 07:48:17.001,2025-03-04 07:52:52.595,Bergen St & Smith St,4446.01,Bergen St & Flatbush Ave,4281.08,40.686744,-73.990632,40.680945,-73.975673,member
2,17F184F80E088790,electric_bike,2025-03-01 20:28:36.021,2025-03-01 20:36:41.340,Bergen St & Smith St,4446.01,Johnson St & Gold St,4668.08,40.686744,-73.990632,40.694749,-73.983625,member
3,241B925B376A39B3,electric_bike,2025-03-05 09:25:53.437,2025-03-05 09:36:50.983,Ave A & E 14 St,5779.11,W 20 St & 7 Ave,6182.02,40.730311,-73.980472,40.742388,-73.997262,member
4,2BCAB8EEAEC26BFB,electric_bike,2025-03-12 17:53:59.432,2025-03-12 17:57:01.529,W 4 St & 7 Ave S,5880.02,King St & Varick St,5687.11,40.734011,-74.002939,40.727897,-74.005363,member


In [3]:
# Guarda el DataFrame combinado en una nueva carpeta 'processed'
df_trips.to_csv('../data/processed/trips_raw_combined.csv', index=False)
print("¡DataFrame guardado correctamente!")

¡DataFrame guardado correctamente!


## 2. Carga datos de estaciones

In [4]:
# Carga datos de estaciones desde la fuente oficial
url_estaciones = 'https://gbfs.citibikenyc.com/gbfs/en/station_information.json'
# Pandas se conecta a la URL, descarga el contenido y, como reconoce que es un archivo de formato JSON, lo lee y lo convierte en un DataFrame.
df_stations_raw = pd.read_json(url_estaciones)
# Se obtiene una serie de pandas donde cada elemento es un diccionario con la información de una estación
# json_normalize toma la lista de diccionario y convierte cada clave en una columna del nuevo DataFrame
df_stations = pd.json_normalize(df_stations_raw['data']['stations'])

df_stations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2307 entries, 0 to 2306
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   eightd_station_services         2307 non-null   object 
 1   lat                             2307 non-null   float64
 2   station_id                      2307 non-null   object 
 3   rental_methods                  2307 non-null   object 
 4   external_id                     2307 non-null   object 
 5   short_name                      2307 non-null   object 
 6   has_kiosk                       2307 non-null   bool   
 7   eightd_has_key_dispenser        2307 non-null   bool   
 8   lon                             2307 non-null   float64
 9   electric_bike_surcharge_waiver  2307 non-null   bool   
 10  capacity                        2307 non-null   int64  
 11  station_type                    2307 non-null   object 
 12  region_id                       22

In [5]:
# Comprobamos las primeras 5 filas de los datos de las estaciones
df_stations.head()

,eightd_station_services,lat,station_id,rental_methods,external_id,short_name,has_kiosk,eightd_has_key_dispenser,lon,electric_bike_surcharge_waiver,capacity,station_type,region_id,name,rental_uris.android,rental_uris.ios
0,[],40.744310,576abdc1-986e-48b0-bde2-051a13d3ff59,"[KEY, CREDITCARD]",576abdc1-986e-48b0-bde2-051a13d3ff59,6218.04,True,False,-73.926010,False,21,classic,71,39 St & Queens Blvd,https://bkn.lft.to/lastmile_qr_scan,https://bkn.lft.to/lastmile_qr_scan
1,[],40.712868,66dc3782-0aca-11e7-82f6-3863bb44ef7c,"[KEY, CREDITCARD]",66dc3782-0aca-11e7-82f6-3863bb44ef7c,5267.08,True,False,-73.956981,False,31,classic,71,Grand St & Havemeyer St,https://bkn.lft.to/lastmile_qr_scan,https://bkn.lft.to/lastmile_qr_scan
2,[],40.633490,2124381939991318658,"[KEY, CREDITCARD]",2124381939991318658,2708.07,False,False,-74.029760,False,0,classic,71,73 St & Ridge Blvd,https://bkn.lft.to/lastmile_qr_scan,https://bkn.lft.to/lastmile_qr_scan
3,[],40.635370,2124035976827702128,"[KEY, CREDITCARD]",2124035976827702128,2793.02,False,False,-74.023420,False,0,classic,71,68 St & 4 Ave,https://bkn.lft.to/lastmile_qr_scan,https://bkn.lft.to/lastmile_qr_scan
4,[],40.868130,6542d952-ca19-410e-9290-ee6b7a6e14cf,"[KEY, CREDITCARD]",6542d952-ca19-410e-9290-ee6b7a6e14cf,8664.06,True,False,-73.884120,False,20,classic,71,Decatur Ave & Bedford Park Blvd,https://bkn.lft.to/lastmile_qr_scan,https://bkn.lft.to/lastmile_qr_scan


## 3. Visualización de ambos DataFrames

In [6]:
# Ver las dimensiones (filas, columnas) de ambos DataFrames
print(f"Dimensiones de Viajes: {df_trips.shape}")
print(f"Dimensiones de Estaciones: {df_stations.shape}")

Dimensiones de Viajes: (3168271, 13)
Dimensiones de Estaciones: (2307, 16)
